# Lily Video Studio — Kaggle T4×2
Use **GPU T4 ×2**, then Run All. The final cell prints a temporary Gradio link.


In [ ]:
!pip -q install -U "diffusers>=0.35.2" "transformers>=4.56.0" "accelerate>=1.2.0" imageio imageio-ffmpeg sentencepiece
print('Install finished. Dependency resolver warnings about unrelated Kaggle packages can be ignored if this cell completes.')


In [ ]:
import torch, gc
from diffusers import DiffusionPipeline

assert torch.cuda.is_available(), 'GPU is off. Kaggle Settings → Accelerator → GPU T4 ×2.'
print('GPUs:', torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

MODEL_ID = 'Wan-AI/Wan2.1-VACE-1.3B-diffusers'
print('Loading Wan… first run downloads a lot of model data.')

try:
    pipe = DiffusionPipeline.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,
        device_map='balanced',
        low_cpu_mem_usage=True,
    )
    print('Loaded across available GPUs.')
except Exception as e:
    print('Balanced load fallback:', repr(e))
    pipe = DiffusionPipeline.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True,
    )
    pipe.enable_model_cpu_offload()

try: pipe.enable_vae_tiling()
except Exception: pass
try: pipe.enable_vae_slicing()
except Exception: pass
gc.collect(); torch.cuda.empty_cache()
print('WAN READY ✨')


In [ ]:
import os, uuid, gc
from PIL import Image
from diffusers.utils import export_to_video

OUT = '/kaggle/working/videos'
os.makedirs(OUT, exist_ok=True)

def _fit(img, w=512, h=320):
    img = img.convert('RGB')
    r, tr = img.width/img.height, w/h
    if r > tr:
        nw = int(img.height*tr); x=(img.width-nw)//2; img=img.crop((x,0,x+nw,img.height))
    else:
        nh = int(img.width/tr); y=(img.height-nh)//2; img=img.crop((0,y,img.width,y+nh))
    return img.resize((w,h), Image.LANCZOS)

def generate_video(image, prompt, steps, seed):
    if image is None: raise ValueError('Upload an image first.')
    if not prompt or not prompt.strip(): raise ValueError('Type a prompt first.')
    image = _fit(image)
    gen = torch.Generator(device='cpu').manual_seed(int(seed)) if int(seed) > 0 else None
    gc.collect(); torch.cuda.empty_cache()
    result = pipe(
        image=image,
        prompt=prompt.strip(),
        height=320,
        width=512,
        num_frames=33,
        num_inference_steps=int(steps),
        guidance_scale=5.0,
        generator=gen,
    ).frames[0]
    path = f'{OUT}/video_{uuid.uuid4().hex[:8]}.mp4'
    export_to_video(result, path, fps=8)
    gc.collect(); torch.cuda.empty_cache()
    return path

print('Generator function ready ✨')


In [ ]:
import gradio as gr

with gr.Blocks() as app:
    gr.Markdown('# ✦ Lily Video Studio')
    gr.Markdown('Your Kaggle GPU is doing the work. Keep this notebook session running.')
    image = gr.Image(type='pil', label='Input image')
    prompt = gr.Textbox(lines=4, label='Motion prompt', placeholder='natural movement, smooth cinematic camera, coherent anatomy')
    with gr.Row():
        subtle = gr.Button('Subtle')
        cinematic = gr.Button('Cinematic')
        dance = gr.Button('Dance')
    subtle.click(lambda: 'subtle breathing, blinking, slight head movement, realistic natural motion, stable camera', outputs=prompt)
    cinematic.click(lambda: 'slow cinematic camera movement, natural body motion, soft hair and fabric movement, smooth coherent motion', outputs=prompt)
    dance.click(lambda: 'energetic dancing, expressive full body movement, dynamic camera, smooth coherent anatomy and motion', outputs=prompt)
    with gr.Accordion('Settings', open=False):
        steps = gr.Slider(6, 20, value=10, step=1, label='Steps (10 recommended)')
        seed = gr.Number(value=0, precision=0, label='Seed (0 = random)')
    go = gr.Button('Generate ✦', variant='primary')
    video = gr.Video(label='Result')
    go.click(generate_video, inputs=[image,prompt,steps,seed], outputs=video)

app.queue(max_size=3).launch(share=True, theme=gr.themes.Soft())
